In [2]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
# 使用DBSCAN聚类分割
print('正在加载点云....')
pcd = o3d.io.read_point_cloud('../data/points/tutorials/room_scan1.pcd')
print(pcd)
o3d.visualization.draw_geometries([pcd])


正在加载点云....
PointCloud with 112586 points.


In [4]:
print('正在DBSCAN聚类')
eps = 0.5
min_points = 1000
with o3d.utility.VerbosityContextManager(o3d.utility.VerbosityLevel.Debug) as cm:
    labels = np.array(pcd.cluster_dbscan(eps, min_points, print_progress=True))
max_label = labels.max()
print(f'point cloud has {max_label + 1} clusters')
colors = plt.get_cmap('tab20')(labels / (max_label if max_label > 0 else 1))
colors[labels < 0] = 0
pcd.colors = o3d.utility.Vector3dVector(colors[:, :3])
o3d.visualization.draw_geometries([pcd])

正在DBSCAN聚类
[Open3D DEBUG] Precompute neighbors.
[Open3D DEBUG] Done Precompute neighbors.
[Open3D DEBUG] Compute Clusters
[Open3D DEBUG] Done Compute Clusters: 3
point cloud has 3 clusters


In [7]:
# RANSAC平面分割
distance_threshhold = 0.5
ransac_n = 5
num_iters = 1000

plane_model, inliers = pcd.segment_plane(distance_threshhold, ransac_n, num_iters)

[a, b, c, d] = plane_model
print(f'Plane equation: {a:.2f}x + {b:.2f}y + {c:.2f}z + {d:.2f} = 0')

inlier_cloud = pcd.select_by_index(inliers)
inlier_cloud.paint_uniform_color([0, 0, 1])
print(inlier_cloud)

outlier_cloud = pcd.select_by_index(inliers, invert=True)
outlier_cloud.paint_uniform_color([1, 0, 0])
print(outlier_cloud)

o3d.visualization.draw_geometries([inlier_cloud, outlier_cloud])

Plane equation: 0.91x + -0.38y + 0.16z + -0.14 = 0
PointCloud with 52270 points.
PointCloud with 60316 points.
